# Homework 6, Q3: Training a Language Model with the Muon Optimizer

We use the same character-level Shakespeare data, the same GPT model (`model.py`), and the same training budget
(`max_iter = 2000`, `batch_size = 12`) as the Lecture 15 notebook. Muon is provided in `muon.py`. It is used like any
PyTorch optimizer, for example `optimizer = Muon(muon_params, lr=muon_lr, weight_decay=0.1)`, where `muon_params`
is the list of parameters that Muon should train (all of them must be 2D) and `muon_lr` is the learning rate you
choose. The other arguments (`momentum=0.95`, `nesterov=True`, `ns_steps=5`) can be left at their defaults.

Fill in the parts marked `# YOUR CODE HERE` and the answer cells marked *Your answer here*. You may create additional cells as needed.

## Setup (same as the Lecture 15 notebook)

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR

from model import GPTConfig, GPT
from muon import Muon


In [ ]:
# Character-level tokenization
with open("data/shakespeare.txt", "r") as f:
    data = f.read()
chars = sorted(list(set(data)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}


def encode(s):
    return [stoi[c] for c in s]


def decode(l):
    return "".join([itos[i] for i in l])


n = len(data)
train_ids = np.array(encode(data[:int(n * 0.9)]), dtype=np.uint16)
val_ids = np.array(encode(data[int(n * 0.9):]), dtype=np.uint16)
print(f"vocab size {vocab_size}, train {len(train_ids):,} tokens, val {len(val_ids):,} tokens")


class TextDataset(Dataset):
    def __init__(self, text_ids, block_size):
        self.data = text_ids
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        chunk = self.data[idx: idx + self.block_size + 1].astype(np.int64)
        return torch.from_numpy(chunk[:-1]), torch.from_numpy(chunk[1:])


block_size = 64
train_dataset = TextDataset(train_ids, block_size)
validate_dataset = TextDataset(val_ids, block_size)


In [ ]:
# Model and training settings from the Lecture 15 notebook
model_args = dict(n_layer=6, n_head=4, n_embd=256, block_size=block_size,
                  bias=True, vocab_size=vocab_size, dropout=0.0)

max_iter = 2000        # total number of training iterations
batch_size = 12
eval_interval = 100    # validation interval
eval_iters = 20        # number of batches to average for the validation loss
warmup_iters = 100     # linear warmup, then cosine decay


def make_model(seed=1337):
    """A freshly initialized model; use the same seed for every run so the runs are comparable."""
    torch.manual_seed(seed)
    return GPT(GPTConfig(**model_args))


## Q3-1: Set up the two optimizers

In [ ]:
model = make_model()
print([pn for pn, p in model.named_parameters()])


In [ ]:
# Split the parameters of the model into a Muon group (the weight matrices of the attention and MLP layers inside
# the transformer blocks) and an AdamW group (all other parameters). Print the number of tensors and the number of
# parameters in each group, and show that every parameter of the model is in exactly one group.

# YOUR CODE HERE


In [ ]:
# Create the two optimizers (Muon from muon.py, and AdamW) and a learning rate scheduler for each
# (warmup + cosine decay to 10% of each optimizer's largest learning rate). Then write the training loop: in every iteration,
# step both optimizers and both schedulers, and zero both gradients. Record the validation loss every
# eval_interval iterations and the training time.
#
# Muon is used like any PyTorch optimizer, e.g. Muon(muon_params, lr=muon_lr, weight_decay=0.1),
# where muon_params is a list of 2D parameters.

# YOUR CODE HERE


## Q3-2: Tune the learning rates

In [ ]:
# AdamW only (as in the Lecture 15 notebook): try at least 3 learning rates.

# YOUR CODE HERE


In [ ]:
# Muon + AdamW: try at least 3 learning rates for Muon and at least 2 learning rates for the AdamW group.

# YOUR CODE HERE


In [ ]:
# Show a table of the best validation loss and the training time of each run. Plot the validation loss curves of
# the best AdamW run and the best Muon + AdamW run on the same axes. Print the best validation loss you achieved.

# YOUR CODE HERE


## Q3-3: Discussion



*Your answer here.*